In [1]:
import os
import io
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torchvision

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Determinism helps reproducibility; does not change core semantics.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(42)



In [2]:
# Paths (keep Kaggle-style paths; choose the one that exists)
BASE1 = Path("../input/cassava-leaf-disease-classification")
BASE2 = Path("/kaggle/input/cassava-leaf-disease-classification")

BASE = BASE1 if BASE1.exists() else BASE2

train_csv_path = BASE / "train.csv"
sample_sub_path = BASE / "sample_submission.csv"
train_img_dir = BASE / "train_images"
test_img_dir = BASE / "test_images"

assert train_csv_path.exists(), f"train.csv not found at {train_csv_path}"
assert sample_sub_path.exists(), f"sample_submission.csv not found at {sample_sub_path}"
assert train_img_dir.exists(), f"train_images not found at {train_img_dir}"
assert test_img_dir.exists(), f"test_images not found at {test_img_dir}"

batch_size = 32
valid_input_size = (512, 682)  # (H, W) for Albumentations Resize

train_df = pd.read_csv(train_csv_path)
sample_sub = pd.read_csv(sample_sub_path)

num_classes = int(train_df["label"].nunique())
assert num_classes == 5, f"Expected 5 classes, got {num_classes}"



In [3]:
# Albumentations v2 compatible transforms (fixes ModuleNotFoundError + Resize not defined)
train_tfms = A.Compose(
    [
        A.Resize(height=valid_input_size[0], width=valid_input_size[1]),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.Normalize(),
        ToTensorV2(),
    ]
)

test_tfms = A.Compose(
    [
        A.Resize(height=valid_input_size[0], width=valid_input_size[1]),
        A.Normalize(),
        ToTensorV2(),
    ]
)


class CassavaTrainDataset(Dataset):
    def __init__(self, df: pd.DataFrame, img_dir: Path, tfms):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.tfms = tfms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.img_dir / row["image_id"]
        img = np.array(Image.open(img_path).convert("RGB"))
        img = self.tfms(image=img)["image"]
        y = int(row["label"])
        return img, y


class CassavaTestDataset(Dataset):
    # IMPORTANT: iterate in sample_submission order to avoid misalignment/length issues
    def __init__(self, image_ids, img_dir: Path, tfms):
        self.image_ids = list(image_ids)
        self.img_dir = img_dir
        self.tfms = tfms

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        img_path = self.img_dir / image_id
        img = np.array(Image.open(img_path).convert("RGB"))
        img = self.tfms(image=img)["image"]
        return image_id, img




In [4]:
# Attempt to load external pickled models if they exist; otherwise train a lightweight fallback model.
model_fnames = [
    "../input/cassava-notebook-15-resnext-models/resnext101wsl_epoch_6.pickle",
    "../input/cassava-notebook-16-models/resnext101wsl_epoch_8.pickle",
]
existing_model_paths = [p for p in model_fnames if Path(p).exists()]

models = []
if len(existing_model_paths) > 0:
    for p in existing_model_paths:
        m = torch.load(p, map_location=device)
        m = m.to(device)
        m.eval()
        models.append(m)

# Fallback: train a standard classifier when pickles are unavailable.
# This keeps the overall "image classifier -> argmax label" semantics and ensures a valid submission is produced.
if len(models) == 0:
    # Simple split (no leakage); minimal and deterministic
    perm = np.random.RandomState(42).permutation(len(train_df))
    split = int(0.9 * len(train_df))
    tr_idx, va_idx = perm[:split], perm[split:]
    tr_df, va_df = train_df.iloc[tr_idx].reset_index(drop=True), train_df.iloc[
        va_idx
    ].reset_index(drop=True)

    train_ds = CassavaTrainDataset(tr_df, train_img_dir, train_tfms)
    valid_ds = CassavaTrainDataset(va_df, train_img_dir, test_tfms)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        drop_last=True,
    )
    valid_loader = DataLoader(
        valid_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        drop_last=False,
    )

    # ResNet18 is available in torchvision; adjust classifier head to 5 classes.
    net = torchvision.models.resnet18(
        weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1
    )
    net.fc = nn.Linear(net.fc.in_features, num_classes)
    net = net.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-2)

    # Keep runtime reasonable; a few epochs over resized images is enough to produce sensible predictions.
    # (No early stopping; fixed epochs.)
    epochs = 2

    for epoch in range(epochs):
        net.train()
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = torch.as_tensor(yb, device=device, dtype=torch.long)
            optimizer.zero_grad(set_to_none=True)
            out = net(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()

        # quick validation accuracy for sanity (doesn't affect submission)
        net.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for xb, yb in valid_loader:
                xb = xb.to(device, non_blocking=True)
                yb = torch.as_tensor(yb, device=device, dtype=torch.long)
                out = net(xb)
                pred = out.argmax(dim=1)
                correct += (pred == yb).sum().item()
                total += yb.numel()
        val_acc = correct / max(1, total)
        print(f"epoch {epoch+1}/{epochs} - val_acc: {val_acc:.4f}")

    net.eval()
    models = [net]




Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 107MB/s]


epoch 1/2 - val_acc: 0.8516
epoch 2/2 - val_acc: 0.8687


In [5]:
class EnsemblePredictor:
    def __init__(self, models):
        self.models = models

    def predict_on_loader(self, loader):
        predictions = []
        filenames = []

        for files, imgs in loader:
            imgs = imgs.to(device, non_blocking=True)
            logits_sum = None
            with torch.no_grad():
                for model in self.models:
                    out = model(imgs)
                    logits_sum = out if logits_sum is None else (logits_sum + out)

            batch_pred = (
                torch.argmax(logits_sum, dim=1)
                .detach()
                .cpu()
                .numpy()
                .astype(int)
                .tolist()
            )
            predictions.extend(batch_pred)
            filenames.extend(list(files))

        return predictions, filenames


test_ds = CassavaTestDataset(sample_sub["image_id"].values, test_img_dir, test_tfms)
test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    drop_last=False,
)

predictor = EnsemblePredictor(models)
predictions, filenames = predictor.predict_on_loader(test_loader)

# Safety checks for valid submission
assert len(predictions) == len(
    sample_sub
), "Prediction count must match sample_submission length"
assert len(filenames) == len(
    sample_sub
), "Filename count must match sample_submission length"
assert set(filenames) == set(
    sample_sub["image_id"].values
), "Filenames must match test set image_ids"



In [6]:
# Write submission in the exact required format and order
sub = pd.DataFrame({"image_id": filenames, "label": predictions})
# Enforce sample_submission ordering (important for Kaggle checks)
sub = sample_sub[["image_id"]].merge(sub, on="image_id", how="left")
assert sub["label"].isna().sum() == 0, "All test images must have predictions"

out_path = "submission.csv"
sub.to_csv(out_path, index=False)
print(f"Wrote {out_path} with shape {sub.shape}")
print(sub.head())

Wrote submission.csv with shape (2676, 2)
         image_id  label
0  1234294272.jpg      3
1  1234332763.jpg      3
2  1234375577.jpg      1
3  1234555380.jpg      1
4  1234571117.jpg      3
